# 交互式叙事与逻辑分析工坊

### 项目概述
本项目是为 "AMD ROCm x LLM 大模型应用开发大赛" 设计的创意写作工具。它不仅是一个 AI 故事续写助手，更是一个严谨的“叙事审计员”。
- **核心技术**：基于 AMD GPU 加速环境，调用 `qwen3-coder:30b` 大模型（32k 上下文）。
- **主要功能**：
  1. **交互式创作**：支持多题材（科幻、悬疑等）的多轮对话续写。
  2. **逻辑一致性审计**：利用大模型作为“专家”，自动检测故事中的逻辑漏洞与情节冲突。
  3. **文学结构分析**：应用“三幕式理论”对内容进行结构化打分与诊断，提升创作质量。
  4. **工程优化**：实现了 32k 超长上下文的动态截断管理与异步非阻塞 UI 交互。


In [ ]:
# 检查 AMD GPU 是否可用
import torch
print(f'ROCm 可用: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "未检测到"}')

In [ ]:
# 你的项目代码从这里开始
import ipywidgets as widgets
from IPython.display import display, HTML
import requests
import json
import threading
import time
import re

# === 1. 全局配置与状态 ===
API_URL = "http://open-webui-ollama.open-webui:11434/api/chat"
MODEL_NAME = "qwen3-coder:30b"
MAX_CHARS_LIMIT = 90000 

story_history = []  # {role, content, type: 'story'/'analysis', story_text, req_text}
stop_event = threading.Event()
is_generating = False # 是否正在等待大模型响应

# === 2. 增强型 UI 样式与 静态滚动锁定脚本 ===
display(HTML("""
<style>
    .story-canvas {
        height: 580px; overflow-y: auto; padding: 25px;
        background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px;
        line-height: 1.8; font-size: 15px; margin-bottom: 15px;
        scroll-behavior: smooth;
    }
    .msg-user-story { border-left: 5px solid #10b981; background: #f0fdf4; padding: 12px; margin-top: 15px; border-radius: 8px; font-weight: 500; }
    .msg-user-req { font-size: 13px; color: #64748b; padding-left: 15px; margin-bottom: 10px; font-style: italic; }
    .msg-bot-story { padding: 5px 0 25px 0; color: #1e293b; white-space: pre-wrap; transition: opacity 0.5s; }
    
    /* 思考中占位样式 */
    .thinking-block { 
        padding: 15px; color: #94a3b8; font-style: italic; 
        background: #f8fafc; border-radius: 8px; border: 1px dashed #e2e8f0;
        margin: 10px 0; animation: pulse 1.5s infinite;
    }
    @keyframes pulse { 0% { opacity: 0.6; } 50% { opacity: 1; } 100% { opacity: 0.6; } }

    /* 分析报告样式 */
    .msg-analysis {
        background: #f8fafc; border: 1px solid #e2e8f0; border-top: 4px solid #3b82f6;
        border-radius: 8px; padding: 20px; margin: 25px 0;
    }
    .msg-analysis h2 { color: #1e3a8a; font-size: 1.2em; border-bottom: 1px solid #cbd5e1; padding-bottom: 5px; margin-top: 15px; }
    .msg-analysis h3 { color: #2563eb; font-size: 1.1em; margin-top: 12px; }
    .score-badge { background: #3b82f6; color: white; padding: 1px 10px; border-radius: 10px; font-weight: bold; }

    .typing-cursor {
        display: inline-block; width: 8px; height: 18px; background: #3b82f6; 
        animation: blink 0.8s infinite; margin-left: 5px; vertical-align: middle;
    }
    @keyframes blink { 0%, 100% { opacity: 1; } 50% { opacity: 0; } }
</style>

<script>
(function() {
    const autoScroll = () => {
        const el = document.querySelector('.story-canvas');
        if (!el) return;
        const threshold = 100;
        const isAtBottom = el.scrollHeight - el.scrollTop - el.clientHeight < threshold;
        
        const observer = new MutationObserver(() => {
            if (isAtBottom) { el.scrollTop = el.scrollHeight; }
        });
        observer.observe(el, { childList: true, subtree: true });
    };

    const init = () => {
        if (document.querySelector('.story-canvas')) autoScroll();
        else setTimeout(init, 500);
    };
    init();
})();
</script>
"""))

# === 3. 核心工具函数：解析与上下文管理 ===

def format_pretty_html(text, mode="story"):
    if mode == "story":
        return text.replace("\n", "<br/>")
    
    # 严格的非 JSON 格式化解析
    text = re.sub(r'^## (.*?)$', r'<h2>\1</h2>', text, flags=re.M)
    text = re.sub(r'^### (.*?)$', r'<h3>\1</h3>', text, flags=re.M)
    text = re.sub(r'(\d+/100)', r'<span class="score-badge">\1</span>', text)
    text = re.sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', text)
    
    # 列表转换
    lines = text.split('\n')
    output = []
    in_list = False
    for line in lines:
        line = line.strip()
        if not line: continue
        if line.startswith(('- ', '* ')):
            if not in_list: output.append('<ul>'); in_list = True
            output.append(f'<li>{line[2:]}</li>')
        else:
            if in_list: output.append('</ul>'); in_list = False
            if not line.startswith('<h'): output.append(f'<p>{line}</p>')
            else: output.append(line)
    if in_list: output.append('</ul>')
    return "".join(output)

def apply_context_management(history):
    total_len = sum(len(m.get('content', '') or m.get('story_text', '')) for m in history)
    if total_len < MAX_CHARS_LIMIT:
        return history
    
    # 保证 System Prompt 和 第一条用户消息（开篇）在位
    head = history[:2]
    # 保留最近 8 条对话
    tail = history[-8:]
    return head + tail

# === 4. UI 组件构建 ===

title_html = widgets.HTML("<h2 style='color:#1e293b; margin-left:10px;'>📖 叙事AI工坊 <span style='font-size:12px;color:#94a3b8;'></span></h2>")
genre_drop = widgets.Dropdown(options=['科幻', '悬疑', '奇幻', '武侠', '极客'], value='科幻', description='📚 题材')
len_drop = widgets.Dropdown(options=[('短 (200字)', 250), ('中 (500字)', 550), ('详尽 (800字)', 850)], value=550, description='📏 篇幅')
controls = widgets.HBox([genre_drop, len_drop])

story_display = widgets.HTML(value="<div class='story-canvas'>等待灵感汇聚...</div>")
story_input = widgets.Textarea(placeholder='在此续写故事片段...', layout={'width':'99%', 'height':'180px'})
req_input = widgets.Textarea(placeholder='（可选）输入剧情指令，例如：BOSS 此时反杀了主角...', layout={'width':'99%', 'height':'45px'})

auto_btn = widgets.Button(description='✍️ 自动续写', button_style='primary', layout={'width':'49%', 'height':'40px'})
guided_btn = widgets.Button(description='🎯 定向续写', button_style='success', layout={'width':'49%', 'height':'40px'})
consistency_btn = widgets.Button(description='🧐 逻辑校验', button_style='info', layout={'width':'32%'})
structure_btn = widgets.Button(description='📊 结构分析', button_style='info', layout={'width':'32%'})
stop_btn = widgets.Button(description='⏸️ 停止', button_style='warning', disabled=True, layout={'width':'15%'})
undo_btn = widgets.Button(description='↩️ 撤销', button_style='danger', layout={'width':'15%'})

app_ui = widgets.VBox([
    title_html, controls, story_display, 
    widgets.VBox([story_input, req_input], layout={'border':'1px solid #e2e8f0','padding':'10px','border_radius':'12px','background':'#fff'}),
    widgets.HBox([auto_btn, guided_btn]),
    widgets.HBox([consistency_btn, structure_btn, stop_btn, undo_btn])
])

def update_ui_canvas():
    """实时渲染历史，如果正在生成则显示“思考中”占位符"""
    html = "<div class='story-canvas'>"
    for m in story_history:
        if m.get("type") == "analysis":
            html += f"<div class='msg-analysis'>{format_pretty_html(m['content'], 'analysis')}</div>"
        else:
            if m["role"] == "user":
                if m.get('story_text'): html += f"<div class='msg-user-story'>【输入】 {m['story_text']}</div>"
                if m.get('req_text'): html += f"<div class='msg-user-req'>指令：{m['req_text']}</div>"
            else:
                html += f"<div class='msg-bot-story'>{format_pretty_html(m['content'])}</div>"
    
    if is_generating:
        # 显示思考中状态
        html += "<div class='thinking-block'>文学灵感涌动中...<span class='typing-cursor'></span></div>"
        
    html += "</div>"
    story_display.value = html

# === 5. 后端非流式调用逻辑 ===

def llm_executor(chat_context, record_type):
    global is_generating
    try:
        # 使用阻塞式 POST 请求
        response = requests.post(API_URL, json={
            "model": MODEL_NAME, 
            "messages": chat_context, 
            "stream": False, # 为了逻辑清晰，此处设为 False
            "options": {"num_ctx": 32768, "temperature": 0.8}
        }, timeout=150)
        
        result = response.json()
        if "message" in result:
            final_content = result["message"].get("content", "")
            if not stop_event.is_set():
                story_history.append({"role": "assistant", "content": final_content, "type": record_type})
    except Exception as e:
        story_history.append({"role": "assistant", "content": f"系统错误: {str(e)}", "type": "analysis"})
    
    is_generating = False
    set_ui_state(False)
    update_ui_canvas()

def trigger_generation(mode="auto"):
    global is_generating
    s_val, r_val = story_input.value.strip(), req_input.value.strip()
    
    if not s_val:
        if not story_history:
            return  # 既没输入，也没历史，才不执行
        else:
            # 如果输入框为空但想续写，我们手动注入一条隐藏的用户指令
            current_req = r_val if r_val else "请紧接上文，继续发展接下来的剧情。"
            story_history.append({
                "role": "user", 
                "type": "story", 
                "story_text": "[程序触发：继续剧情]", 
                "req_text": current_req
            })
    else:
        # 正常输入模式
        story_history.append({
            "role": "user", 
            "type": "story", 
            "story_text": s_val, 
            "req_text": r_val if mode=="guided" else ""
        })
    story_input.value, req_input.value = "", ""
    
    is_generating = True
    stop_event.clear()
    set_ui_state(True)
    update_ui_canvas()

    # 应用 32k 截断
    safe_history = apply_context_management(story_history)
    
    # 创作类提示词 (2.0 风格)
    sys_prompt = f"你是一流的{genre_drop.value}小说家。请模仿前文笔触续写{len_drop.value}字左右。要求情感细腻，情节逻辑严密。"
    
    messages = [{"role": "system", "content": sys_prompt}]
    for m in safe_history:
        if m.get("type") == "story":
            content = m["content"] if m["role"]=="assistant" else f"正文：{m.get('story_text','')}\n要求：{m.get('req_text','')}"
            messages.append({"role": m["role"], "content": content})
            
    threading.Thread(target=llm_executor, args=(messages, "story")).start()

def trigger_analysis(a_type):
    global is_generating
    if not story_history: return
    
    is_generating = True
    stop_event.clear()
    set_ui_state(True)
    update_ui_canvas()

    pure_text = "\n".join([m.get("content", "") if m["role"]=="assistant" else m.get("story_text", "") for m in story_history if m.get("type")=="story"])
    
    if a_type == "consistency":
        prompt = f"作为资深逻辑审计专家，请核查以下情节。禁止输出JSON，按格式汇报：\n## 1. 关键要素清单\n## 2. 潜在逻辑漏洞引发的冲突\n## 3. 严谨度评分\n\n内容：{pure_text[-5000:]}"
    else:
        prompt = f"作为文学博士，请对以下作品进行三幕式结构分析。禁止输出JSON，按格式汇报：\n## 整体评分：[分数]/100\n## 1. 结构化诊断\n## 2. 节奏评价\n## 3. 建议\n\n内容：{pure_text[:7000]}"

    threading.Thread(target=llm_executor, args=([{"role": "system", "content": "严谨的文学分析助手。"},{"role": "user", "content": prompt}], "analysis")).start()

def set_ui_state(gen):
    for b in [auto_btn, guided_btn, consistency_btn, structure_btn, undo_btn]: b.disabled = gen
    stop_btn.disabled = not gen

# === 6. 绑定事件与显示 ===
auto_btn.on_click(lambda b: trigger_generation("auto"))
guided_btn.on_click(lambda b: trigger_generation("guided"))
consistency_btn.on_click(lambda b: trigger_analysis("consistency"))
structure_btn.on_click(lambda b: trigger_analysis("structure"))
stop_btn.on_click(lambda b: stop_event.set())
undo_btn.on_click(lambda b: (story_history.pop() if story_history else None, update_ui_canvas()))

display(app_ui)